<a href="https://colab.research.google.com/github/hmnaz213/native/blob/main/ICUMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# I apologize for the previous delay.
# Please share your data or problem description here, and I'll do my best to solve it efficiently!

In [2]:
import pandas as pd
import io
import re

raw_data = """No. Trim Level Year of Manufacture Make Model HDV Currency Origin Code HS Code Excange Rate Receipt Date Assessment Date CIF NCY Total Tax
1 S 2016 NISSAN SENTRA 16,780 USD US 8703232000 5.8365 27/08/2021 25/08/2021 38,740.24 13,964.54
2 SV 2016 NISSAN SENTRA 18,550 USD US 8703232000 5.8032 27/07/2021 26/07/2021 42,145.74 15,193.95
3 SV 2016 NISSAN SENTRA 18,550 USD US 8703232000 5.7544 16/06/2021 09/06/2021 41,791.33 15,066.18
4 SR 2016 NISSAN SENTRA 20,410 USD US 8703232000 5.7445 07/06/2021 04/06/2021 45,491.79 16,401.97
5 S 2016 NISSAN SENTRA 16,780 USD US 8703232000 5.7499 08/06/2021 20/05/2021 38,165.42 13,757.35
"""

# Because spaces are used both as delimiters and inside text (e.g., 'Trim Level', 'NISSAN SENTRA'),
# a standard pd.read_csv with space delimiter will misalign columns.
# Here is a robust way to parse it by extracting known patterns.

lines = raw_data.strip().split('\n')
headers = ['No.', 'Trim Level', 'Year', 'Make', 'Model', 'HDV', 'Currency', 'Origin', 'HS Code', 'Exchange Rate', 'Receipt Date', 'Assessment Date', 'CIF NCY', 'Total Tax']

parsed_data = []
for line in lines[1:]: # Skip header row
    # Split from the right to isolate the numerical and date columns safely
    parts = line.split()
    if len(parts) < 14:
        continue

    # The last 9 columns are consistently single unbroken strings
    tail = parts[-9:]

    # The first column is 'No.'
    no = parts[0]

    # The rest in the middle belongs to Trim, Year, Make, Model
    # We know Year is typically a 4 digit number, let's find it to anchor
    middle = parts[1:-9]
    year_idx = next(i for i, val in enumerate(middle) if val.isdigit() and len(val) == 4)

    trim_level = " ".join(middle[:year_idx])
    year = middle[year_idx]
    make = middle[year_idx+1]
    model = " ".join(middle[year_idx+2:])

    row = [no, trim_level, year, make, model] + tail
    parsed_data.append(row)

df = pd.DataFrame(parsed_data, columns=headers)
display(df.head())

,No.,Trim Level,Year,Make,Model,HDV,Currency,Origin,HS Code,Exchange Rate,Receipt Date,Assessment Date,CIF NCY,Total Tax
0,1,S,2016,NISSAN,SENTRA,"16,780",USD,US,8703232000,5.8365,27/08/2021,25/08/2021,"38,740.24","13,964.54"
1,2,SV,2016,NISSAN,SENTRA,"18,550",USD,US,8703232000,5.8032,27/07/2021,26/07/2021,"42,145.74","15,193.95"
2,3,SV,2016,NISSAN,SENTRA,"18,550",USD,US,8703232000,5.7544,16/06/2021,09/06/2021,"41,791.33","15,066.18"
3,4,SR,2016,NISSAN,SENTRA,"20,410",USD,US,8703232000,5.7445,07/06/2021,04/06/2021,"45,491.79","16,401.97"
4,5,S,2016,NISSAN,SENTRA,"16,780",USD,US,8703232000,5.7499,08/06/2021,20/05/2021,"38,165.42","13,757.35"
